# Solution: DDP gradient sync

Сначала проверяем глобальные инварианты ранков, затем независимо обрабатываем каждый параметр. Такой порядок даёт понятные ошибки до арифметики и отражает требование DDP: все ранки должны участвовать в одной и той же коллективной операции с совместимыми тензорами.

In [ ]:
import torch


def ddp_gradient_sync(
    rank_grads: list[dict[str, torch.Tensor | None]],
) -> dict[str, torch.Tensor | None]:
    if not rank_grads:
        raise ValueError("rank_grads must contain at least one rank")

    expected_keys = set(rank_grads[0])
    if any(set(rank_grad) != expected_keys for rank_grad in rank_grads[1:]):
        raise ValueError("All ranks must have the same parameter keys")

    synchronized: dict[str, torch.Tensor | None] = {}
    for name in rank_grads[0]:
        gradients = [rank_grad[name] for rank_grad in rank_grads]
        missing = [gradient is None for gradient in gradients]

        # A frozen parameter has no gradient on every rank.
        if all(missing):
            synchronized[name] = None
            continue
        if any(missing):
            raise ValueError(f"Gradient {name!r} must be None on all ranks or none")

        tensors = [gradient for gradient in gradients if gradient is not None]
        expected_shape = tensors[0].shape
        if any(tensor.shape != expected_shape for tensor in tensors[1:]):
            raise ValueError(f"Gradient {name!r} has inconsistent shapes")

        # Accumulating low-precision gradients in fp32 avoids avoidable rounding.
        output_dtype = tensors[0].dtype
        accumulator_dtype = (
            torch.float32
            if output_dtype in (torch.float16, torch.bfloat16)
            else output_dtype
        )
        synchronized[name] = (
            torch.stack([tensor.to(accumulator_dtype) for tensor in tensors])
            .mean(dim=0)
            .to(output_dtype)
        )

    return synchronized


Деление на число ранков сохраняет масштаб градиента, который получился бы на объединённом батче при обычном усреднении loss. Приведение fp16/bf16 к float32 выполняется только на время редукции: выход сохраняет исходные dtype и shape.

In [ ]:
from torch_judge import check

check('ddp_gradient_sync')
